# LUT-SLM adapter inference on Colab

Upload an image, enter a color-grading instruction, and run `ericrcwu/LUT_SLM_sft_adapters/bl_63cd1bf7_smokefull` on a Colab GPU. The notebook downloads only that adapter folder plus the official Qwen base, reconstructs the required 151,924-token vocabulary **in memory**, emits 64 LUT-code tokens, decodes them with the immutable frozen tokenizer, renders the graded image, and evaluates it with the metrics used by AceTone.

Before running:

1. Choose **Runtime → Change runtime type → GPU** (T4, L4, or A100).
2. Add a Colab secret named `HF_TOKEN` with read access to the private adapter repository and enable notebook access. If it is absent, the notebook opens the Hugging Face login widget.
3. Edit `PROMPT` in the configuration cell, then use **Runtime → Run all**. Upload the original image and, for reference metrics, its ground-truth graded target when prompted.

> The notebook loads the frozen VQ-VAE read-only through `tokenizer.frozen`; it never enables or calls `eval/lut_decoder.py`, and never retrains, re-gates, or modifies the tokenizer. It also does **not** rerun the unseeded `sft.vocab_resize` job.


## Paper-aligned evaluation

[AceTone §5.2](https://arxiv.org/html/2604.00530v1#S5.SS2) evaluates decoded, LUT-applied images with **DeQA aesthetic score** (higher is better), **PSNR** (higher), **LPIPS-AlexNet** (lower), and mean **CIEDE2000 ΔE** (lower). This notebook implements those definitions and also records inference latency, syntax validity, the decoded `.cube`, and an input/output preview.

Important scope: this is a **single-sample, paper-aligned diagnostic**. PSNR, LPIPS, and ΔE require a paired ground-truth graded image. The paper reports averages over PST-50, AceTone-Bench-Transfer (1,024 samples), and AceTone-Bench-Instruct (128 samples), so one uploaded example is not directly comparable to its tables. The paper's Gemini-2.5-Pro average rank needs four anonymized competing outputs, and its human preference study needs participants; neither is fabricated here.


## Stage the immutable decoder

The adapter upload contains the VLM adapter and LUT vocabulary, while the canonical frozen VQ-VAE lives in `ericrcwu/LUT_SLM`. The next cell pins that dataset revision, range-downloads only the tail of its final uncompressed tar shard, extracts `tokenizer/final/model.pt` and `manifest.json`, and verifies the codebook, encoder, and decoder hashes. It does not download the 10 GB corpus or alter any decoder weights.


In [ ]:
%pip -q install -U "transformers>=4.49,<5" "peft==0.19.1" "accelerate>=1.2,<2" "bitsandbytes>=0.45" "qwen-vl-utils>=0.0.10" "huggingface_hub>=0.28" "safetensors>=0.4" "scipy>=1.11" "scikit-image>=0.22" "lpips>=0.1.4"


In [ ]:
#@title Configuration
REPO_ID = "ericrcwu/LUT_SLM_sft_adapters"
ADAPTER_SUBFOLDER = "bl_63cd1bf7_smokefull"
BASE_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
LOCAL_ADAPTER_ROOT = "/content/LUT_SLM_sft_adapters"

PROMPT = "Make the image warmer with gentle contrast and slightly richer colors." #@param {type:"string"}
CONSTRAINED_DECODING = True #@param {type:"boolean"}
UPLOAD_GROUND_TRUTH = True #@param {type:"boolean"}
RUN_DEQA = True #@param {type:"boolean"}
SEED = 1234

# These values are bound into this adapter's manifest.
MIN_PIXELS = 3136
MAX_PIXELS = 200704
EXPECTED_VOCAB_SIZE = 151924
EXPECTED_TOKENIZER_VERSION = "vq_v2_srgbres_17to4_cb256_t64__w91cffdd2c82f"
SLM_REPO = "https://github.com/ericrcwu001/SLM.git"
SLM_COMMIT = "2b6043f0ea8ed32d1abe3a967b25cc60536cf1ab"
CORPUS_REPO_ID = "ericrcwu/LUT_SLM"
CORPUS_REVISION = "f555cfd6a99f1ee4cb675a0da9b68504a1773e39"
print({"prompt": PROMPT, "constrained": CONSTRAINED_DECODING, "ground_truth": UPLOAD_GROUND_TRUTH, "deqa": RUN_DEQA})


In [ ]:
import json
import os
from pathlib import Path

import torch
from huggingface_hub import login, notebook_login, snapshot_download

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is attached. In Colab choose Runtime > Change runtime type > GPU.")

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} ({gpu.total_memory / 2**30:.1f} GiB)")

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    print("HF_TOKEN secret not found; use the login widget with an account that can read the private repo.")
    notebook_login()

snapshot_download(
    repo_id=REPO_ID,
    allow_patterns=[f"{ADAPTER_SUBFOLDER}/*"],
    local_dir=LOCAL_ADAPTER_ROOT,
)
adapter_dir = Path(LOCAL_ADAPTER_ROOT) / ADAPTER_SUBFOLDER
required = ["adapter_config.json", "adapter_manifest.json", "adapter_model.safetensors", "tokenizer.json", "tokenizer_config.json"]
missing = [name for name in required if not (adapter_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"Adapter download is incomplete; missing: {missing}")

adapter_manifest = json.loads((adapter_dir / "adapter_manifest.json").read_text())
adapter_config = json.loads((adapter_dir / "adapter_config.json").read_text())
assert adapter_manifest["base_model_id"] == BASE_MODEL_ID
assert adapter_manifest["vocab_size_after_resize"] == EXPECTED_VOCAB_SIZE
assert adapter_manifest["tokenizer_version"] == EXPECTED_TOKENIZER_VERSION
assert set(adapter_config["modules_to_save"]) == {"embed_tokens", "lm_head"}
print(json.dumps({
    "adapter_dir": str(adapter_dir),
    "adapter_sha256": adapter_manifest["adapter_sha256"],
    "adapter_step": adapter_manifest["adapter_step"],
    "tokenizer_version": adapter_manifest["tokenizer_version"],
}, indent=2))


In [ ]:
import requests
import shutil
import subprocess
import sys
from huggingface_hub import hf_hub_download, hf_hub_url

repo_dir = Path("/content/SLM")
if not (repo_dir / ".git").is_dir():
    subprocess.run(["git", "clone", "--filter=blob:none", SLM_REPO, str(repo_dir)], check=True)
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", SLM_COMMIT], check=True)
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", SLM_COMMIT], check=True)

staged_final = Path("/content/slm/tokenizer/final")
staged_final.mkdir(parents=True, exist_ok=True)
staged_model = staged_final / "model.pt"
staged_manifest_path = staged_final / "manifest.json"

if not (staged_model.is_file() and staged_manifest_path.is_file()):
    # The corpus is sharded into uncompressed tars. tokenizer/final sorts last, so fetch
    # only the final 32 MiB with HTTP Range instead of downloading the 1.9 GB shard.
    stage_manifest_local = hf_hub_download(
        repo_id=CORPUS_REPO_ID, filename="stage_manifest.json", repo_type="dataset",
        revision=CORPUS_REVISION,
    )
    corpus_stage_manifest = json.loads(Path(stage_manifest_local).read_text())
    if corpus_stage_manifest.get("compression") != "none":
        raise RuntimeError("Range extraction requires the pinned uncompressed corpus revision.")
    final_shard = corpus_stage_manifest["shards"][-1]
    archive_size = int(final_shard["bytes"])
    tail_bytes = min(32 * 1024 * 1024, archive_size)
    range_start = ((archive_size - tail_bytes) // 512) * 512
    shard_url = hf_hub_url(
        repo_id=CORPUS_REPO_ID, filename=final_shard["name"], repo_type="dataset",
        revision=CORPUS_REVISION,
    )
    range_headers = {"Range": f"bytes={range_start}-{archive_size - 1}"}
    if hf_token:
        range_headers["Authorization"] = f"Bearer {hf_token}"
    with requests.get(shard_url, headers=range_headers, stream=True, timeout=120) as response:
        if response.status_code != 206:
            raise RuntimeError(
                f"Expected HTTP 206 for bounded decoder fetch, got {response.status_code}; "
                "refusing to download the full shard."
            )
        tail_blob = response.content
    expected_tail_size = archive_size - range_start
    if len(tail_blob) != expected_tail_size:
        raise RuntimeError(f"Short ranged download: {len(tail_blob)} != {expected_tail_size}")

    def extract_tail_tar_member(blob, member_name):
        # Tar headers and payload padding are 512-byte aligned. Validate each candidate
        # header checksum before trusting its name or size.
        for offset in range(0, len(blob) - 511, 512):
            header = blob[offset:offset + 512]
            if header[257:262] != b"ustar":
                continue
            try:
                stored_checksum = int(header[148:156].rstrip(b"\0 " ).strip() or b"0", 8)
            except ValueError:
                continue
            computed_checksum = sum(header[:148]) + 8 * 32 + sum(header[156:])
            if stored_checksum != computed_checksum:
                continue
            name = header[:100].split(b"\0", 1)[0].decode("utf-8")
            prefix = header[345:500].split(b"\0", 1)[0].decode("utf-8")
            full_name = f"{prefix}/{name}" if prefix else name
            if full_name != member_name:
                continue
            size = int(header[124:136].rstrip(b"\0 " ).strip() or b"0", 8)
            payload_start = offset + 512
            payload_end = payload_start + size
            if payload_end > len(blob):
                raise RuntimeError(f"Tar member {member_name} is truncated in ranged response.")
            return bytes(blob[payload_start:payload_end])
        raise FileNotFoundError(f"{member_name} not found in tail of {final_shard['name']}")

    staged_model.write_bytes(extract_tail_tar_member(tail_blob, "tokenizer/final/model.pt"))
    staged_manifest_path.write_bytes(extract_tail_tar_member(tail_blob, "tokenizer/final/manifest.json"))
    del tail_blob
    print(f"Fetched frozen decoder from {CORPUS_REPO_ID}@{CORPUS_REVISION[:12]} via {expected_tail_size / 2**20:.1f} MiB range.")
else:
    print(f"Reusing staged frozen decoder at {staged_final}")

os.environ["SLM_ARTIFACT_ROOT"] = "/content/slm"  # lowercase is intentional
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
from tokenizer.frozen import load_frozen_vqvae

frozen_vqvae, frozen_manifest = load_frozen_vqvae(str(staged_final))
assert frozen_manifest["tokenizer_version"] == EXPECTED_TOKENIZER_VERSION
assert frozen_manifest["vq_codebook_sha256"] == adapter_manifest["vq_codebook_sha256"]
print(json.dumps({
    "tokenizer_version": frozen_manifest["tokenizer_version"],
    "vq_codebook_sha256": frozen_manifest["vq_codebook_sha256"],
    "vq_decoder_sha256": frozen_manifest["vq_decoder_sha256"],
    "verified": True,
}, indent=2))


In [ ]:
import gc
from transformers import AutoProcessor, AutoTokenizer, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from peft import PeftModel

# Reconstruct the exact tokenizer IDs without reading the adapter's legacy
# extra_special_tokens list (new Transformers releases expect that field to be a dict).
lut_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer_base_vocab_size = len(lut_tokenizer)
canonical_lut_tokens = ["<lut_bos>", "<lut_eos>", "<unsupported>"] + [
    f"<lut_{index:03d}>" for index in range(256)
]
added_count = lut_tokenizer.add_special_tokens({"additional_special_tokens": canonical_lut_tokens})
assert added_count == 259, f"Expected to add 259 LUT tokens, added {added_count}"
assert len(lut_tokenizer) == EXPECTED_VOCAB_SIZE, (len(lut_tokenizer), EXPECTED_VOCAB_SIZE)
actual_lut_ids = [lut_tokenizer.convert_tokens_to_ids(token) for token in canonical_lut_tokens]
expected_lut_ids = list(range(tokenizer_base_vocab_size, tokenizer_base_vocab_size + 259))
assert actual_lut_ids == expected_lut_ids, "Canonical LUT token IDs are not the contiguous vocab tail"
assert all(
    lut_tokenizer.encode(token, add_special_tokens=False) == [token_id]
    for token, token_id in zip(canonical_lut_tokens, actual_lut_ids)
), "At least one LUT token is not atomic"

processor = AutoProcessor.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
processor.tokenizer = lut_tokenizer

major, _minor = torch.cuda.get_device_capability(0)
compute_dtype = torch.bfloat16 if major >= 8 else torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=compute_dtype,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

# Shape-only resize. PEFT then restores the trained full embed_tokens/lm_head from
# modules_to_save, so the random temporary rows are never used for inference.
base_vocab_size = base.get_input_embeddings().num_embeddings
if base_vocab_size != len(lut_tokenizer):
    base.resize_token_embeddings(len(lut_tokenizer), mean_resizing=False)

model = PeftModel.from_pretrained(
    base,
    adapter_dir,
    is_trainable=False,
    autocast_adapter_dtype=False,
    low_cpu_mem_usage=True,
)
model.eval()
del base
gc.collect()

assert model.get_input_embeddings().num_embeddings == EXPECTED_VOCAB_SIZE
print(f"Loaded base vocab {base_vocab_size} -> LUT vocab {len(lut_tokenizer)} using {compute_dtype}.")
print(f"CUDA allocated: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")


In [ ]:
#@title Upload one input image
from google.colab import files
from IPython.display import display
from PIL import Image

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No image was uploaded.")
image_name = next(iter(uploaded))
image_path = str(Path("/content") / image_name)
Path(image_path).write_bytes(uploaded[image_name])

with Image.open(image_path) as image:
    image.verify()
with Image.open(image_path) as image:
    print(f"Image: {image_name} — {image.size[0]}×{image.size[1]} {image.mode}")
    display(image.copy())
print(f"Prompt: {PROMPT}")


In [ ]:
#@title Upload the paired ground-truth graded image (paper reference metrics)
ground_truth_path = None
if UPLOAD_GROUND_TRUTH:
    print("Upload the ground-truth graded version of the same input image. Its pixel dimensions must match.")
    target_upload = files.upload()
    if not target_upload:
        raise RuntimeError("UPLOAD_GROUND_TRUTH=True but no target image was uploaded.")
    ground_truth_name = next(iter(target_upload))
    ground_truth_path = str(Path("/content") / f"ground_truth_{Path(ground_truth_name).name}")
    Path(ground_truth_path).write_bytes(target_upload[ground_truth_name])
    with Image.open(ground_truth_path) as target_image:
        target_image.verify()
    with Image.open(ground_truth_path) as target_image, Image.open(image_path) as source_image:
        if target_image.size != source_image.size:
            raise ValueError(f"Ground truth {target_image.size} must match input {source_image.size}; no silent resize is performed.")
        print(f"Ground truth: {ground_truth_name} — {target_image.size[0]}×{target_image.size[1]}")
        display(target_image.copy())
else:
    print("No ground truth requested: PSNR, LPIPS, and ΔE will be reported as unavailable.")


In [ ]:
import re
import time
from qwen_vl_utils import process_vision_info
from transformers import LogitsProcessor, LogitsProcessorList

LUT_BOS = "<lut_bos>"
LUT_EOS = "<lut_eos>"
UNSUPPORTED = "<unsupported>"
CODE_TOKENS = [f"<lut_{i:03d}>" for i in range(256)]
bos_id = lut_tokenizer.convert_tokens_to_ids(LUT_BOS)
lut_eos_id = lut_tokenizer.convert_tokens_to_ids(LUT_EOS)
unsupported_id = lut_tokenizer.convert_tokens_to_ids(UNSUPPORTED)
code_ids = [lut_tokenizer.convert_tokens_to_ids(token) for token in CODE_TOKENS]
assert len(set(code_ids)) == 256 and all(token_id >= 0 for token_id in code_ids)
code_id_to_value = {token_id: value for value, token_id in enumerate(code_ids)}

class LutGrammarLogitsProcessor(LogitsProcessor):
    """Batch-size-one grammar: unsupported OR BOS + exactly 64 codes + EOS."""
    def __init__(self, prompt_length):
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores):
        generated = input_ids.shape[1] - self.prompt_length
        if generated == 0:
            allowed = [bos_id, unsupported_id]
        else:
            first = int(input_ids[0, self.prompt_length])
            if first == unsupported_id:
                allowed = [unsupported_id]
            elif 1 <= generated <= 64:
                allowed = code_ids
            elif generated == 65:
                allowed = [lut_eos_id]
            else:
                allowed = [lut_tokenizer.eos_token_id]
        masked = torch.full_like(scores, float("-inf"))
        masked[:, allowed] = scores[:, allowed]
        return masked

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image_path},
        {"type": "text", "text": PROMPT},
    ],
}]
prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[prompt_text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = {name: tensor.to("cuda:0") for name, tensor in inputs.items()}
prompt_length = inputs["input_ids"].shape[1]
logits_processors = LogitsProcessorList([LutGrammarLogitsProcessor(prompt_length)]) if CONSTRAINED_DECODING else None

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
started = time.perf_counter()
with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=66 if CONSTRAINED_DECODING else 67,
        do_sample=False,
        num_beams=1,
        use_cache=True,
        logits_processor=logits_processors,
        eos_token_id=[lut_tokenizer.eos_token_id, lut_eos_id, unsupported_id],
        pad_token_id=lut_tokenizer.eos_token_id,
    )
torch.cuda.synchronize()
elapsed_seconds = time.perf_counter() - started

generated_ids = output_ids[0, prompt_length:].tolist()
generated_tokens = lut_tokenizer.convert_ids_to_tokens(generated_ids)
raw_decoded_text = lut_tokenizer.decode(
    generated_ids, skip_special_tokens=False, clean_up_tokenization_spaces=False
).strip()

if generated_ids == [unsupported_id]:
    output_kind = "unsupported"
    code_values = []
elif (len(generated_ids) == 66 and generated_ids[0] == bos_id and
      generated_ids[-1] == lut_eos_id and all(token_id in code_id_to_value for token_id in generated_ids[1:-1])):
    output_kind = "lut_tokens"
    code_values = [code_id_to_value[token_id] for token_id in generated_ids[1:-1]]
else:
    output_kind = "invalid"
    code_values = [code_id_to_value[token_id] for token_id in generated_ids if token_id in code_id_to_value]

normalized_token_text = " ".join(generated_tokens)
tokens_per_second = len(generated_ids) / elapsed_seconds
peak_cuda_gib = torch.cuda.max_memory_allocated() / 2**30

print(f"kind={output_kind} generated_tokens={len(generated_ids)} code_tokens={len(code_values)}")
print(f"inference={elapsed_seconds:.2f}s ({tokens_per_second:.2f} generated tokens/s), peak CUDA={peak_cuda_gib:.2f} GiB")
print("\nNormalized output:")
print(normalized_token_text)
if raw_decoded_text != normalized_token_text:
    print("\nRaw tokenizer decode:")
    print(raw_decoded_text)


In [ ]:
#@title Decode the 64 codes, apply the LUT, and render artifacts
import numpy as np
from PIL import ImageOps
from data_pipeline.lut_ops import apply_lut_trilinear
from eval.cube_io import cube_bytes_hash, identity_grid, write_cube

artifact_dir = Path("/content/lut_slm_inference_result")
if artifact_dir.exists():
    shutil.rmtree(artifact_dir)
artifact_dir.mkdir(parents=True)
graded_path = None
cube_path = None
preview_path = None
absolute_lut = None
cube_sha256 = None

with Image.open(image_path) as opened_source:
    source_rgb = ImageOps.exif_transpose(opened_source).convert("RGB")
source_rgb.save(artifact_dir / "input.png")

if output_kind == "lut_tokens":
    if len(code_values) != 64:
        raise RuntimeError(f"Refusing to decode {len(code_values)} codes; exactly 64 are required.")
    residual_lut = frozen_vqvae.decode(code_values)
    absolute_lut = np.clip(identity_grid(17) + residual_lut, 0.0, 1.0)
    source_array = np.asarray(source_rgb, dtype=np.float64) / 255.0
    graded_array = np.clip(apply_lut_trilinear(absolute_lut, source_array), 0.0, 1.0)
    # AceTone's released evaluation converts with multiplication followed by uint8 cast.
    graded_image = Image.fromarray((graded_array * 255.0).astype(np.uint8), mode="RGB")
    graded_path = str(artifact_dir / "graded.png")
    graded_image.save(graded_path)
    cube_path = str(artifact_dir / "output.cube")
    write_cube(cube_path, absolute_lut)
    cube_sha256 = cube_bytes_hash(absolute_lut)

    panels = [source_rgb, graded_image]
    if ground_truth_path:
        with Image.open(ground_truth_path) as opened_target:
            panels.append(ImageOps.exif_transpose(opened_target).convert("RGB"))
    preview = Image.new("RGB", (source_rgb.width * len(panels), source_rgb.height))
    for panel_index, panel in enumerate(panels):
        preview.paste(panel, (panel_index * source_rgb.width, 0))
    preview_path = str(artifact_dir / "preview_input_pred_target.png")
    preview.save(preview_path)
    print(f"Decoded LUT: {cube_path} (sha256={cube_sha256})")
    display(preview)
else:
    print(f"No LUT rendered because output_kind={output_kind!r}. No identity fallback is used.")


In [ ]:
#@title AceTone reference metrics: PSNR, LPIPS-AlexNet, and mean CIEDE2000 ΔE
from skimage import color
from skimage.metrics import peak_signal_noise_ratio
import lpips
from torchvision.transforms.functional import to_tensor

paper_metrics = {
    "deqa_aesthetic_0_1": None,
    "deqa_raw_0_5": None,
    "psnr_db": None,
    "lpips_alex": None,
    "mean_ciede2000": None,
}
metric_status = {
    "reference_metrics": "unavailable_without_valid_lut_and_ground_truth",
    "deqa": "pending" if RUN_DEQA else "disabled",
}

if graded_path and ground_truth_path:
    with Image.open(graded_path) as pred_open, Image.open(ground_truth_path) as gt_open:
        pred_image = pred_open.convert("RGB")
        gt_image = gt_open.convert("RGB")
    if pred_image.size != gt_image.size:
        raise ValueError(f"Prediction {pred_image.size} and ground truth {gt_image.size} differ.")

    pred_u8 = np.asarray(pred_image, dtype=np.uint8)
    gt_u8 = np.asarray(gt_image, dtype=np.uint8)
    paper_metrics["psnr_db"] = float(peak_signal_noise_ratio(gt_u8, pred_u8, data_range=255))
    pred_lab = color.rgb2lab(pred_u8.astype(np.float32) / 255.0)
    gt_lab = color.rgb2lab(gt_u8.astype(np.float32) / 255.0)
    paper_metrics["mean_ciede2000"] = float(np.mean(color.deltaE_ciede2000(gt_lab, pred_lab)))

    lpips_model = lpips.LPIPS(net="alex").to("cuda:0").eval()
    pred_tensor = (to_tensor(pred_image).unsqueeze(0).to("cuda:0") * 2.0) - 1.0
    gt_tensor = (to_tensor(gt_image).unsqueeze(0).to("cuda:0") * 2.0) - 1.0
    with torch.inference_mode():
        paper_metrics["lpips_alex"] = float(lpips_model(gt_tensor, pred_tensor).item())
    del lpips_model, pred_tensor, gt_tensor
    torch.cuda.empty_cache()
    metric_status["reference_metrics"] = "computed_single_sample"

print(json.dumps({"paper_metrics": paper_metrics, "status": metric_status}, indent=2))


### DeQA aesthetic score

AceTone uses the public `zhiyuanyou/DeQA-Score-Mix3` model and divides its native 0–5 output by five. That checkpoint is an 8.2B model (about 16.4 GB of safetensors) whose published loader requires Transformers 4.46.3, while Qwen2.5-VL needs a newer release. To avoid dependency contamination, the next cell frees the generation model and runs DeQA in an isolated virtual environment. Exact full-precision DeQA is skipped on GPUs below 20 GiB; use an L4/A100 for the complete path.


In [ ]:
#@title AceTone aesthetic metric (DeQA; runs last because it unloads Qwen)
deqa_status = metric_status["deqa"]
if RUN_DEQA and not graded_path:
    deqa_status = "skipped_no_graded_image"
elif RUN_DEQA and gpu.total_memory / 2**30 < 20.0:
    deqa_status = "skipped_exact_model_requires_gpu_with_at_least_20_GiB"
elif RUN_DEQA:
    # Release the Qwen/PEFT GPU allocation before the 8.2B DeQA model is loaded.
    del model, inputs, output_ids
    gc.collect()
    torch.cuda.empty_cache()

    deqa_venv = Path("/content/deqa_venv")
    deqa_python = deqa_venv / "bin/python"
    if not deqa_python.is_file():
        subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(deqa_venv)], check=True)
        subprocess.run([str(deqa_python), "-m", "pip", "install", "-q", "transformers==4.46.3", "accelerate>=0.26,<2", "sentencepiece"], check=True)

    deqa_script = Path("/content/run_deqa_score.py")
    deqa_script.write_text("""
import json
import sys
from pathlib import Path
import torch
from PIL import Image
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(
    'zhiyuanyou/DeQA-Score-Mix3',
    trust_remote_code=True,
    attn_implementation='eager',
    torch_dtype=torch.float16,
    device_map={'': 0},
    low_cpu_mem_usage=True,
    use_safetensors=True,
)
model.eval()
with Image.open(sys.argv[1]) as opened:
    image = opened.convert('RGB')
with torch.inference_mode():
    score = model.score([image])
if torch.is_tensor(score):
    raw = float(score.reshape(-1)[0].item())
else:
    raw = float(score[0])
Path(sys.argv[2]).write_text(json.dumps({'raw_0_5': raw, 'normalized_0_1': raw / 5.0}))
""", encoding="utf-8")
    deqa_output = Path("/content/deqa_score.json")
    subprocess.run([str(deqa_python), str(deqa_script), graded_path, str(deqa_output)], check=True)
    deqa_result = json.loads(deqa_output.read_text())
    paper_metrics["deqa_raw_0_5"] = deqa_result["raw_0_5"]
    paper_metrics["deqa_aesthetic_0_1"] = deqa_result["normalized_0_1"]
    deqa_status = "computed_single_sample"

metric_status["deqa"] = deqa_status
print(json.dumps({"paper_metrics": paper_metrics, "status": metric_status}, indent=2))


In [ ]:
#@title Save and download the inference result
result_dir = artifact_dir
result_dir.mkdir(parents=True, exist_ok=True)

if ground_truth_path:
    shutil.copy2(ground_truth_path, result_dir / "ground_truth.png")
(result_dir / "output_tokens.txt").write_text(normalized_token_text + "\n", encoding="utf-8")
result = {
    "evaluation_protocol": {
        "name": "AceTone Section 5.2 single-sample aligned",
        "paper": "https://arxiv.org/html/2604.00530v1#S5.SS2",
        "released_code": "https://github.com/martian422/AceTone/blob/open-source-ready/eval/predict_lut_ddp.py",
        "not_directly_comparable_to_published_dataset_averages": True,
    },
    "repo_id": REPO_ID,
    "adapter_subfolder": ADAPTER_SUBFOLDER,
    "adapter_sha256": adapter_manifest["adapter_sha256"],
    "base_model_id": BASE_MODEL_ID,
    "tokenizer_version": adapter_manifest["tokenizer_version"],
    "vq_codebook_sha256": frozen_manifest["vq_codebook_sha256"],
    "vq_decoder_sha256": frozen_manifest["vq_decoder_sha256"],
    "image": Path(image_path).name,
    "ground_truth_supplied": bool(ground_truth_path),
    "prompt": PROMPT,
    "constrained_decoding": CONSTRAINED_DECODING,
    "seed": SEED,
    "output_kind": output_kind,
    "output_token_text": normalized_token_text,
    "raw_decoded_text": raw_decoded_text,
    "generated_token_ids": generated_ids,
    "lut_code_values": code_values,
    "cube_sha256": cube_sha256,
    "paper_metrics": paper_metrics,
    "metric_status": metric_status,
    "elapsed_seconds": elapsed_seconds,
    "generated_tokens_per_second": tokens_per_second,
    "peak_cuda_gib": peak_cuda_gib,
    "gpu": gpu.name,
}
(result_dir / "inference_result.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
(result_dir / "paper_metrics.json").write_text(json.dumps({"metrics": paper_metrics, "status": metric_status}, indent=2) + "\n", encoding="utf-8")
archive = shutil.make_archive("/content/lut_slm_inference_result", "zip", result_dir)
print(f"Wrote {archive}")
files.download(archive)


## Try another instruction or image

Set `CONSTRAINED_DECODING = False` to inspect completely free generation; malformed output is reported as `invalid` rather than repaired. If DeQA ran, the Qwen model was deliberately unloaded to free GPU memory; rerun the model-loading cell before trying another prompt. If DeQA was disabled or skipped, change `PROMPT` and rerun from the inference cell.
